# 07 — TopK SAE Training (DeiT / DINOv2)

Train a Sparse Autoencoder on patch-token activations extracted in `06_collect_activations.ipynb`.

**Architecture**: TopK SAE — `d_sae = 768 × 16 = 12 288` features, `k = 32` active per token.  
**Loss**: MSE reconstruction only (no L1 penalty — TopK enforces exact sparsity).  
**Input**: HDF5 file produced by notebook 06 (`*_layer9_acts.h5`).

## 0. Kaggle Setup

In [ ]:
!rm -rf /kaggle/working/xai-vit-medical
!git clone https://github.com/youssef-nouiouar/xai-vit-medical.git /kaggle/working/xai-vit-medical
!pip install -q h5py
!pip install -q PyDrive2

## Google Drive

In [ ]:
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

## Paths

Point `H5_PATH` to the HDF5 file produced by notebook 06.

In [ ]:
import os

# ← update this path to your HDF5 file (Kaggle dataset input)
H5_PATH  = '/kaggle/input/activations/deit_base_patch16_layer9_acts.h5'

SAVE_DIR = '/kaggle/working/sae_checkpoints'
os.makedirs(SAVE_DIR, exist_ok=True)

## 1. Imports & Config

All hyperparameters in one place. To switch to DINOv2, update `MODEL_LABEL` and `H5_PATH` above.

In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import h5py
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')

# ── SAE Architecture ──────────────────────────────────────────────────────────
D_INPUT   = 768
EXPANSION = 16
D_SAE     = D_INPUT * EXPANSION   # 12 288
K         = 32                    # features active per token

# ── Training ──────────────────────────────────────────────────────────────────
LR           = 3e-4
BETA1        = 0.9
BETA2        = 0.999
WEIGHT_DECAY = 0.0
BATCH_SIZE   = 4096    # tokens per batch (not images)
N_EPOCHS     = 5
WARMUP_STEPS = 1000
VAL_RATIO    = 0.1
SAVE_EVERY   = 5000    # steps between intermediate checkpoints

# Optional: limit number of images to reduce RAM usage (None = use all)
MAX_IMAGES   = None

# ── Label ─────────────────────────────────────────────────────────────────────
MODEL_LABEL = 'deit_base_patch16'   # used in checkpoint filenames

print(f'd_input={D_INPUT}  d_sae={D_SAE} ({EXPANSION}x)  k={K}')
print(f'LR={LR}  batch_size={BATCH_SIZE} tokens  epochs={N_EPOCHS}')

## 2. Load Activations

The H5 file contains `(N_images, 196, 768)` float16 activations from correctly classified images only.  
We flatten to a flat token array `(N_images × 196, 768)` and split train / val at the token level.

In [ ]:
print(f'Loading: {H5_PATH}')

with h5py.File(H5_PATH, 'r') as f:
    acts_h5   = f['activations'][:MAX_IMAGES]   # (N, 196, 768) float16
    labels_h5 = f['labels'][:MAX_IMAGES]
    layer_idx = int(f.attrs.get('layer', 9))
    print(f'  Shape     : {acts_h5.shape}  dtype={acts_h5.dtype}')
    print(f'  Layer     : {layer_idx}')
    print(f'  RAM usage : {acts_h5.nbytes / 1e9:.2f} GB')

# Flatten: (N, 196, 768) → (N*196, 768)  — no copy, just a view
N, P, D = acts_h5.shape
tokens  = acts_h5.reshape(N * P, D)   # float16

# Shuffle and split
rng  = np.random.default_rng(SEED)
perm = rng.permutation(len(tokens))
cut  = int(len(perm) * (1 - VAL_RATIO))

train_tokens = tokens[perm[:cut]]
val_tokens   = tokens[perm[cut:]]

print(f'\nTrain tokens : {len(train_tokens):,}')
print(f'Val   tokens : {len(val_tokens):,}')

## 3. Normalization Stats & DataLoaders

Compute per-dimension mean and std on a random subset of training tokens.  
Applied on-the-fly in the dataset `__getitem__`.

In [ ]:
# Compute stats on up to 200k tokens (faster, representative enough)
NORM_SAMPLES = min(200_000, len(train_tokens))
subset = train_tokens[:NORM_SAMPLES].astype(np.float32)
MEAN   = torch.from_numpy(subset.mean(axis=0)).float()              # (768,)
STD    = torch.from_numpy(subset.std(axis=0).clip(1e-8)).float()    # (768,)
print(f'mean abs avg : {MEAN.abs().mean():.4f}')
print(f'std avg      : {STD.mean():.4f}')


class SAETokenDataset(Dataset):
    def __init__(self, tokens_fp16, mean, std):
        self.tokens = tokens_fp16    # (N, 768) float16 numpy
        self.mean   = mean
        self.std    = std

    def __len__(self):
        return len(self.tokens)

    def __getitem__(self, i):
        x = torch.from_numpy(self.tokens[i].astype(np.float32))
        return (x - self.mean) / self.std


train_dataset = SAETokenDataset(train_tokens, MEAN, STD)
val_dataset   = SAETokenDataset(val_tokens,   MEAN, STD)

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                           shuffle=True,  num_workers=2, pin_memory=True, drop_last=True)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                           shuffle=False, num_workers=2, pin_memory=True)

print(f'\nTrain batches : {len(train_loader)}')
print(f'Val   batches : {len(val_loader)}')

## 4. SAE Architecture

TopK SAE (Gao 2024):  
```
acts  = TopK( W_enc @ (x − b_dec) + b_enc,  k )   # encoder + sparsity
x_hat = W_dec @ acts + b_dec                        # decoder
loss  = MSE(x, x_hat)
```
Decoder rows are kept at unit L2 norm after every optimizer step.

In [ ]:
class TopKSAE(nn.Module):

    def __init__(self, d_input: int, d_sae: int, k: int):
        super().__init__()
        self.d_input = d_input
        self.d_sae   = d_sae
        self.k       = k

        self.W_enc = nn.Parameter(torch.empty(d_input, d_sae))
        self.b_enc = nn.Parameter(torch.zeros(d_sae))
        self.W_dec = nn.Parameter(torch.empty(d_sae,  d_input))
        self.b_dec = nn.Parameter(torch.zeros(d_input))

        nn.init.kaiming_uniform_(self.W_enc)
        nn.init.kaiming_uniform_(self.W_dec)
        self.normalize_decoder()

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        pre  = (x - self.b_dec) @ self.W_enc + self.b_enc   # (B, d_sae)
        vals, idx = pre.topk(self.k, dim=-1)
        acts = torch.zeros_like(pre)
        acts.scatter_(-1, idx, vals.clamp(min=0))            # only positive activations
        return acts

    def decode(self, acts: torch.Tensor) -> torch.Tensor:
        return acts @ self.W_dec + self.b_dec

    def forward(self, x: torch.Tensor):
        acts  = self.encode(x)
        x_hat = self.decode(acts)
        return x_hat, acts

    @torch.no_grad()
    def normalize_decoder(self):
        """Unit-norm each row of W_dec (one row = one feature direction)."""
        norms = self.W_dec.norm(dim=1, keepdim=True).clamp(min=1e-8)
        self.W_dec.data.div_(norms)


sae = TopKSAE(D_INPUT, D_SAE, K).to(DEVICE)
n_params = sum(p.numel() for p in sae.parameters())
print(f'SAE params   : {n_params:,}')
print(f'  W_enc      : {tuple(sae.W_enc.shape)}')
print(f'  W_dec      : {tuple(sae.W_dec.shape)}')
print(f'  L0 (exact) : {K}  ({K/D_SAE*100:.2f}% of features per token)')

## 5. Training

In [ ]:
@torch.no_grad()
def evaluate_sae(sae, loader, device):
    sae.eval()
    total_mse = 0.0
    ss_res    = 0.0
    ss_tot    = 0.0
    ever_fired = torch.zeros(sae.d_sae, device=device)
    n = 0

    for x in loader:
        x = x.to(device)
        x_hat, acts = sae(x)

        total_mse += F.mse_loss(x_hat, x, reduction='sum').item()
        ss_res    += ((x - x_hat) ** 2).sum().item()
        ss_tot    += ((x - x.mean(0, keepdim=True)) ** 2).sum().item()
        ever_fired += (acts > 0).sum(dim=0).float()
        n += len(x)

    avg_mse  = total_mse / (n * sae.d_input)
    r2       = 1.0 - ss_res / (ss_tot + 1e-8)
    dead_pct = (ever_fired == 0).float().mean().item() * 100
    sae.train()
    return avg_mse, r2, dead_pct

In [ ]:
def save_checkpoint(sae, optimizer, step, history, config, tag):
    fname = f'sae_{config["model_label"]}_layer{config["layer_idx"]}_{tag}.pt'
    path  = os.path.join(config['save_dir'], fname)
    ckpt  = {
        'step':    step,
        'W_enc':   sae.W_enc.data.cpu(),
        'b_enc':   sae.b_enc.data.cpu(),
        'W_dec':   sae.W_dec.data.cpu(),
        'b_dec':   sae.b_dec.data.cpu(),
        'mean':    MEAN,
        'std':     STD,
        'history': history,
        'config':  config,
    }
    if optimizer is not None:
        ckpt['optimizer'] = optimizer.state_dict()
    torch.save(ckpt, path)
    return path

In [ ]:
def train_sae(sae, train_loader, val_loader, config, device):
    optimizer = torch.optim.Adam(
        sae.parameters(),
        lr=config['lr'],
        betas=(config['beta1'], config['beta2']),
        weight_decay=config['weight_decay'],
    )

    total_steps = config['n_epochs'] * len(train_loader)

    def lr_lambda(step):
        if step < config['warmup_steps']:
            return step / max(config['warmup_steps'], 1)
        t = (step - config['warmup_steps']) / max(total_steps - config['warmup_steps'], 1)
        return 0.5 * (1 + math.cos(math.pi * t))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    history   = {'train_loss': [], 'val_mse': [], 'val_r2': [], 'dead_pct': []}
    best_val  = float('inf')
    step      = 0

    for epoch in range(1, config['n_epochs'] + 1):
        sae.train()
        epoch_loss = 0.0

        pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{config["n_epochs"]}')
        for x in pbar:
            x = x.to(device)

            optimizer.zero_grad(set_to_none=True)
            x_hat, acts = sae(x)
            loss = F.mse_loss(x_hat, x)
            loss.backward()
            optimizer.step()
            scheduler.step()
            sae.normalize_decoder()   # enforce unit-norm after every step

            epoch_loss += loss.item()
            step       += 1

            pbar.set_postfix(loss=f'{loss.item():.4f}',
                             lr=f'{scheduler.get_last_lr()[0]:.2e}')

            if step % config['save_every'] == 0:
                save_checkpoint(sae, optimizer, step, history, config,
                                tag=f'step{step}')

        val_mse, val_r2, dead_pct = evaluate_sae(sae, val_loader, device)
        avg_train = epoch_loss / len(train_loader)
        history['train_loss'].append(avg_train)
        history['val_mse'].append(val_mse)
        history['val_r2'].append(val_r2)
        history['dead_pct'].append(dead_pct)

        print(f'  Epoch {epoch} | train={avg_train:.4f} | val={val_mse:.4f} '
              f'| R²={val_r2:.4f} | dead={dead_pct:.1f}%')

        if val_mse < best_val:
            best_val = val_mse
            save_checkpoint(sae, optimizer, step, history, config, tag='best')
            print(f'  ★  new best (val_mse={val_mse:.6f})')

    print(f'\nDone. Best val_mse={best_val:.6f}')
    return history

## 6. Run Training

In [ ]:
config = {
    'model_label':   MODEL_LABEL,
    'layer_idx':     layer_idx,
    'd_input':       D_INPUT,
    'd_sae':         D_SAE,
    'k':             K,
    'expansion':     EXPANSION,
    'lr':            LR,
    'beta1':         BETA1,
    'beta2':         BETA2,
    'weight_decay':  WEIGHT_DECAY,
    'batch_size':    BATCH_SIZE,
    'n_epochs':      N_EPOCHS,
    'warmup_steps':  WARMUP_STEPS,
    'save_every':    SAVE_EVERY,
    'save_dir':      SAVE_DIR,
}

print('Config:')
for k, v in config.items():
    if k != 'save_dir':
        print(f'  {k:<16}: {v}')

sae     = TopKSAE(D_INPUT, D_SAE, K).to(DEVICE)
history = train_sae(sae, train_loader, val_loader, config, DEVICE)

## 7. Plots

| Metric | Healthy target |
|--------|---------------|
| MSE (val) | Decreasing, stabilizing |
| R² (val) | > 0.85 |
| Dead features | 10 – 30 % |

In [ ]:
epochs = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epochs, history['train_loss'], label='Train')
axes[0].plot(epochs, history['val_mse'],    label='Val')
axes[0].set(xlabel='Epoch', ylabel='MSE', title='Reconstruction Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs, history['val_r2'], color='green')
axes[1].axhline(0.85, color='red', linestyle='--', label='target R²=0.85')
axes[1].set(xlabel='Epoch', ylabel='R²', title='Explained Variance (val)')
axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(epochs, history['dead_pct'], color='orange')
axes[2].axhline(30, color='red',   linestyle='--', label='30% (upper healthy)')
axes[2].axhline(10, color='green', linestyle='--', label='10% (lower healthy)')
axes[2].set(xlabel='Epoch', ylabel='Dead features (%)', title='Dead Features')
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
plot_path = os.path.join(SAVE_DIR, f'sae_{MODEL_LABEL}_training.png')
plt.savefig(plot_path, dpi=150)
plt.show()

## 8. Save Final Checkpoint & Upload

In [ ]:
final_path = save_checkpoint(sae, None, -1, history, config, tag='final')
print(f'Final checkpoint : {final_path}')

# Upload best, final, and plot to Google Drive
to_upload = [
    os.path.join(SAVE_DIR, f'sae_{MODEL_LABEL}_layer{layer_idx}_best.pt'),
    os.path.join(SAVE_DIR, f'sae_{MODEL_LABEL}_layer{layer_idx}_final.pt'),
    plot_path,
]

for fpath in to_upload:
    if os.path.exists(fpath):
        gd = drive.CreateFile({'title': os.path.basename(fpath)})
        gd.SetContentFile(fpath)
        gd.Upload()
        print(f'Uploaded : {os.path.basename(fpath)}  (id={gd["id"]})')
    else:
        print(f'Skipped  : {os.path.basename(fpath)} (not found)')